# Logistic Regression and Model Evaluation — Worked Solutions (Python)

The materials are based on the applied exercises in *An Introduction to Statistical Learning with Applications in Python* ([ISLP](https://www.statlearning.com/)), Chapter 5 (Resampling Methods). We fit a **logistic regression** classifier and estimate its **test error** with the **validation-set approach**, then use the **bootstrap** to obtain standard errors of the coefficients and compare them with the classical `glm` formula.

**Data:** `Default` — 10,000 credit-card customers, used to predict whether a customer *defaults* on their debt.

In [24]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.utils import resample

pd.set_option("display.width", 120)

## Question 1 — Predicting default status on credit-card payment

The dataset has 10,000 observations on 4 variables:

* **default** — factor (No/Yes): did the customer default on the debt?
* **student** — factor (No/Yes): is the customer a student?
* **balance** — average balance remaining on the credit card after the monthly payment.
* **income** — income of the customer.

We load the data and encode the two factors as 0/1 dummies (`Yes = 1`). `default01` is our binary response.

In [25]:
Default = pd.read_csv('/Users/chipi/Desktop/MONASH/Sem2_26/FIT5149/material/w2/Week 2 Applied Class Questions-20260806/default.csv')
Default["default01"] = (Default["default"] == "Yes").astype(int)
Default["student01"] = (Default["student"] == "Yes").astype(int)
Default.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   default    10000 non-null  object 
 1   student    10000 non-null  object 
 2   balance    10000 non-null  float64
 3   income     10000 non-null  float64
 4   default01  10000 non-null  int64  
 5   student01  10000 non-null  int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 468.9+ KB


Numerical summary of the variables. Note the response is highly **imbalanced** — only about 3.3% of customers default, which we must keep in mind when reading any error rate.

In [26]:
print(Default[["balance", "income"]].describe().round(2))
print("\ndefault value counts:\n", Default["default"].value_counts())
print("\nstudent value counts:\n", Default["student"].value_counts())
print("\nbase (null) error rate if we always predict 'No':",
      round(Default["default01"].mean(), 4))

        balance    income
count  10000.00  10000.00
mean     835.37  33516.98
std      483.71  13336.64
min        0.00    771.97
25%      481.73  21340.46
50%      823.64  34552.64
75%     1166.31  43807.73
max     2654.32  73554.23

default value counts:
 default
No     9667
Yes     333
Name: count, dtype: int64

student value counts:
 student
No     7056
Yes    2944
Name: count, dtype: int64

base (null) error rate if we always predict 'No': 0.0333


### 1.1 Fit a logistic regression using `income` and `balance` to predict `default`

The logistic-regression model is
$$\Pr(\text{default}=1\mid X)=\frac{e^{\beta_0+\beta_1\,\text{income}+\beta_2\,\text{balance}}}{1+e^{\beta_0+\beta_1\,\text{income}+\beta_2\,\text{balance}}}.$$
We estimate it by maximum likelihood with `statsmodels` GLM (Binomial family, logit link).

In [27]:
def fit_logit(df, cols):
    """Fit a binomial-logit GLM of default01 on `cols`; returns the fitted result."""
    X = sm.add_constant(df[cols].astype(float))
    y = df["default01"].values
    return sm.GLM(y, X, family=sm.families.Binomial()).fit()

model = fit_logit(Default, ["income", "balance"])
print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                      y   No. Observations:                10000
Model:                            GLM   Df Residuals:                     9997
Model Family:                Binomial   Df Model:                            2
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -789.48
Date:                Fri, 07 Aug 2026   Deviance:                       1579.0
Time:                        11:43:21   Pearson chi2:                 6.95e+03
No. Iterations:                     9   Pseudo R-squ. (CS):             0.1256
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -11.5405      0.435    -26.544      0.0

**How to read this output**

* **Coefficients (`coef`).** In log-odds. A one-unit rise in `balance` raises the log-odds of default by $\hat\beta_2 = 0.0056$, i.e. multiplies the odds by $e^{0.0056}\approx 1.0056$ (about +0.56% odds per extra dollar of balance — a large effect over a realistic \$1,000 swing). `income` has a tiny positive coefficient ($2.08\times10^{-5}$) because income is on a scale of tens of thousands.
* **Std err / z / P>|z|.** The **Wald z-statistic** is $\hat\beta_j/\text{SE}(\hat\beta_j)$; it tests $H_0:\beta_j=0$. Both `balance` (z ≈ 24.8) and `income` (z ≈ 4.2) are significant at the 0.1% level, so both predictors matter, though `balance` dominates.
* **Deviance & Log-Likelihood.** Deviance $=-2\log L = 1579.0$ is a *badness-of-fit* measure (lower is better). Log-likelihood is $-789.48$.

We compare the model against a **null model** (intercept only) with a likelihood-ratio (chi-squared) test. The statistic is `null deviance − residual deviance`, distributed $\chi^2$ with degrees of freedom equal to the number of predictors added (here 2).

In [28]:
LR_stat = model.null_deviance - model.deviance
df_diff = int(model.df_model)
from scipy import stats
p_value = stats.chi2.sf(LR_stat, df_diff)
print(f"Null deviance     : {model.null_deviance:.2f}")
print(f"Residual deviance : {model.deviance:.2f}")
print(f"LR chi-square stat: {LR_stat:.2f}  on  df = {df_diff}")
print(f"p-value           : {p_value:.3e}")

Null deviance     : 2920.65
Residual deviance : 1578.97
LR chi-square stat: 1341.68  on  df = 2
p-value           : 4.541e-292


The likelihood-ratio statistic is huge (≈ 1341.7 on 2 df, p ≈ 4.5e−292), so the two-predictor model fits **dramatically** better than the intercept-only model.

### 1.2 Estimate the test error with the validation-set approach

Steps:
1. Split the sample into a **training set** and a **validation set** (here 50/50).
2. Fit the multiple logistic regression on the *training* observations only.
3. For each validation observation, compute the posterior probability of default and classify as **Yes** if that probability exceeds **0.5**.
4. The **validation error** is the fraction of misclassified validation observations.

In [29]:
def validation_error(cols, train_idx, valid_idx):
    m = fit_logit(Default.iloc[train_idx], cols)
    Xv = sm.add_constant(Default.iloc[valid_idx][cols].astype(float))
    prob = m.predict(Xv)
    pred = (prob > 0.5).astype(int)
    return np.mean(pred != Default.iloc[valid_idx]["default01"].values)

np.random.seed(1)
n = len(Default)
idx = np.random.permutation(n)
train_idx, valid_idx = idx[: n // 2], idx[n // 2 :]

err_1_2 = validation_error(["income", "balance"], train_idx, valid_idx)
print(f"Validation-set error (income + balance): {err_1_2:.4f}")

Validation-set error (income + balance): 0.0282


The estimated test error is about **2.8%**. Because roughly 3.3% of customers default, a trivial classifier that always predicts *No* would already achieve ~3.3% error — so the model beats the naive baseline, but on this imbalanced data the raw error rate is a weak headline metric (a confusion matrix / sensitivity would tell the fuller story).

### 1.3 Repeat with three different splits and comment

Here the splits are formed from the `income` variable (rather than at random), producing three different training subsets of roughly 2/3 of the data each. We report the validation error for each.

In [30]:
quantiles = np.percentile(Default["income"], [33, 67])
print("33rd and 67th percentiles of income:", np.round(quantiles, 2))

33rd and 67th percentiles of income: [25309.38 40732.33]


In [31]:
train1 = (Default["income"] > 25309.38)
train2 = (Default["income"] < 25309.38) | (Default["income"] > 40732.33)
train3 = (Default["income"] < 40732.33)

In [32]:
for name, mask in [("train1", train1), ("train2", train2), ("train3", train3)]:
    tr = np.where(mask.values)[0]
    va = np.where(~mask.values)[0]
    err = validation_error(["income", "balance"], tr, va)
    print(f"{name}: train n = {mask.sum():>4d},  valid n = {(~mask).sum():>4d},"
          f"  validation error = {err:.4f}")

train1: train n = 6700,  valid n = 3300,  validation error = 0.0370
train2: train n = 6600,  valid n = 3400,  validation error = 0.0232
train3: train n = 6700,  valid n = 3300,  validation error = 0.0236


**Comment.** The three validation errors are **0.037, 0.023 and 0.024** — all in the low-single-digit range, confirming the estimate is broadly stable, but they vary more than random splits would.

*Critical note.* These are **systematic (non-random) splits on `income`**, so training and validation sets have different income distributions (distribution shift). `train1` trains only on higher-income customers and is then validated on lower-income customers, which is why its error (3.7%) is the highest. The **correct** validation-set procedure uses **random** splits so that training and validation sets are exchangeable; random 50/50 splits on this data give errors clustered near 2.4–2.8% with less spread. The reason the estimate survives even a biased split here is that `income` is a weak predictor of default (all the signal is in `balance`), so shifting the income distribution barely changes the fit.

### 1.4 Add a dummy for `student`; does it reduce the test error?

We refit with `income`, `balance` and the `student` dummy, using the **same** train/validation split as in 1.2 so the comparison is fair.

In [33]:
err_1_4 = validation_error(["income", "balance", "student01"], train_idx, valid_idx)
print(f"Validation error (income + balance)          : {err_1_2:.4f}")
print(f"Validation error (income + balance + student): {err_1_4:.4f}")
print(f"Change from adding student: {err_1_4 - err_1_2:+.4f}")

Validation error (income + balance)          : 0.0282
Validation error (income + balance + student): 0.0282
Change from adding student: +0.0000


**Comment.** Adding the `student` dummy leaves the validation error essentially unchanged (≈ 2.8% either way). So on the basis of the validation-set approach, **including `student` does not reduce the test error rate**. `student` does carry some information about default (students tend to default more, holding balance fixed), but once `balance` is in the model the extra dummy does not move the 0.5-threshold classifications, so test error does not improve.

## Question 2 — Bootstrap standard errors

We now estimate the standard errors of the `income` and `balance` coefficients in two ways: (1) the classical `glm` formula, and (2) the **bootstrap**. A random seed is set for reproducibility.

### 2.1 Standard errors from the GLM formula

In [34]:
model = fit_logit(Default, ["income", "balance"])
se = model.bse
print("Coefficient estimates:")
print(model.params.rename({"const": "b0(intercept)", "income": "b1(income)",
                           "balance": "b2(balance)"}).to_string())
print("\nClassical GLM standard errors:")
print(se.rename({"const": "SE b0", "income": "SE b1", "balance": "SE b2"}).to_string())

Coefficient estimates:
b0(intercept)   -11.540468
b1(income)        0.000021
b2(balance)       0.005647

Classical GLM standard errors:
SE b0    0.434772
SE b1    0.000005
SE b2    0.000227


The classical standard errors are **SE(β0) ≈ 0.4348**, **SE(β1, income) ≈ 4.99e−06**, **SE(β2, balance) ≈ 2.27e−04**. These come from the inverse Fisher information and rely on the model being correctly specified and on large-sample (asymptotic) normality.

### 2.2 Write `boot_fn`

`boot_fn(data, index)` takes the data and a vector of row indices and returns the fitted coefficient estimates for the `income + balance` logistic model on that resample.

In [35]:
def boot_fn(data, index):
    m = fit_logit(data.iloc[index], ["income", "balance"])
    return m.params.values   # array: [b0, b1(income), b2(balance)]

# sanity check on the full sample (index = all rows) -> matches 2.1
print(np.round(boot_fn(Default, np.arange(len(Default))), 6))

[-1.1540468e+01  2.1000000e-05  5.6470000e-03]


### 2.3 Bootstrap the standard errors using `resample`

In [36]:
B = 1000
n = len(Default)
coefs = np.zeros((B, 3))
for b in range(B):
    idx_b = resample(np.arange(n), replace=True, n_samples=n, random_state=b)
    coefs[b] = boot_fn(Default, idx_b)

boot_se = coefs.std(axis=0, ddof=1)
labels = ["b0 (intercept)", "b1 (income)", "b2 (balance)"]
print(f"Bootstrap with B = {B} resamples\n")
print(f"{'coef':<16}{'bootstrap mean':>16}{'bootstrap SE':>16}{'GLM SE':>16}")
for j, lab in enumerate(labels):
    print(f"{lab:<16}{coefs[:, j].mean():>16.6g}{boot_se[j]:>16.6g}{se.values[j]:>16.6g}")

Bootstrap with B = 1000 resamples

coef              bootstrap mean    bootstrap SE          GLM SE
b0 (intercept)          -11.5847        0.445314        0.434772
b1 (income)          2.09602e-05     4.93533e-06     4.98525e-06
b2 (balance)          0.00567103     0.000231714     0.000227381


The bootstrap standard errors are **SE(β0) ≈ 0.45**, **SE(β1) ≈ 5.0e−06**, **SE(β2) ≈ 2.3e−04**.

### 2.4 Comment on the two sets of standard errors

**Comment.** The bootstrap and GLM standard errors are **very close** (they agree to about two significant figures for all three coefficients). This is reassuring:

* The classical `glm` SEs are derived from the inverse Fisher information and assume (a) the logistic model is the correct data-generating process and (b) the large-sample normal approximation holds.
* The **bootstrap makes neither assumption** — it re-estimates variability directly by resampling the observed data with replacement. When the two agree, it tells us the parametric assumptions behind the `glm` formula are reasonable for this dataset and sample size (n = 10,000). Bootstrap SEs are stochastic, so they will change slightly with the seed or the number of replicates $B$, but not enough to alter any conclusion.